In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

## 1. Load the IMDB Dataset

The IMDB dataset contains 50,000 movie reviews split evenly into training and testing sets. Each review is labeled as positive (1) or negative (0).

We load the dataset in its raw integer-encoded form first, then decode it back to text so we can work with `TextVectorization`.

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
# Load IMDB dataset (integer-encoded)
# num_words=None loads the full vocabulary
(x_train_enc, y_train), (x_test_enc, y_test) = keras.datasets.imdb.load_data()

print(f"Training samples: {len(x_train_enc)}")
print(f"Test samples: {len(x_test_enc)}")
print(f"Label distribution (train): {np.bincount(y_train)}")

In [ ]:
# Decode integer sequences back to text
word_index = keras.datasets.imdb.get_word_index()

# Reverse the word index: integer -> word
# The indices are offset by 3 because 0=padding, 1=start, 2=unknown, 3=unused
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<pad>"
reverse_word_index[1] = "<start>"
reverse_word_index[2] = "<unk>"
reverse_word_index[3] = "<unused>"

def decode_review(encoded_review):
    """Convert an integer-encoded review back to a text string."""
    return " ".join(reverse_word_index.get(i, "?") for i in encoded_review)

# Decode all reviews to text
x_train_text = np.array([decode_review(seq) for seq in x_train_enc])
x_test_text = np.array([decode_review(seq) for seq in x_test_enc])

print(f"\nSample review (first 300 chars):")
print(x_train_text[0][:300])
print(f"\nLabel: {'Positive' if y_train[0] == 1 else 'Negative'}")

In [ ]:
# Explore a few samples
for i in range(3):
    sentiment = "Positive" if y_train[i] == 1 else "Negative"
    preview = x_train_text[i][:150]
    print(f"Review {i} ({sentiment}): {preview}...\n")

## 2. Configure TextVectorization

The `TextVectorization` layer handles:
- Lowercasing and stripping punctuation (by default)
- Building a vocabulary from the training data
- Converting text strings to fixed-length integer sequences

Key parameters:
- `max_tokens=10000` -- keep only the 10,000 most frequent words
- `output_sequence_length=200` -- pad or truncate all sequences to length 200

In [ ]:
# Create and adapt the TextVectorization layer
MAX_TOKENS = 10000
MAX_LENGTH = 200

text_vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_sequence_length=MAX_LENGTH,
    output_mode="int",
)

# Adapt (build vocabulary) on the training data only
text_vectorizer.adapt(x_train_text)

# Inspect the vocabulary
vocab = text_vectorizer.get_vocabulary()
print(f"Vocabulary size: {len(vocab)}")
print(f"First 20 tokens: {vocab[:20]}")
print(f"Last 10 tokens: {vocab[-10:]}")

## 3. Demo: Text to Integers and Back

Let us see how `TextVectorization` transforms a raw text string into an integer sequence, and how we can reverse the process.

In [ ]:
# Forward pass: raw text -> integer sequence
sample_text = "This movie was absolutely wonderful and I loved every minute of it"
encoded = text_vectorizer([sample_text])

print(f"Raw text: {sample_text}")
print(f"\nEncoded (first 20 values): {encoded[0][:20].numpy()}")
print(f"Sequence length: {encoded.shape[1]}")

In [ ]:
# Reverse: integer sequence -> text
def decode_vectorized(int_sequence, vocabulary):
    """Convert an integer sequence from TextVectorization back to text."""
    return " ".join(
        vocabulary[idx] for idx in int_sequence if idx != 0  # skip padding
    )

decoded = decode_vectorized(encoded[0].numpy(), vocab)
print(f"Decoded text: {decoded}")
print(f"\nOriginal:     {sample_text.lower()}")

In [ ]:
# Visualize the token-to-index mapping
words = sample_text.lower().split()
print(f"{'Word':<20} {'Index':>6}")
print("-" * 28)
for word in words:
    idx = vocab.index(word) if word in vocab else 1  # 1 = OOV
    print(f"{word:<20} {idx:>6}")

## 4. Embedding Layer

The `Embedding` layer creates a lookup table that maps each integer token to a dense vector of a fixed size (here, 128 dimensions). These vectors are learned during training.

- Input shape: `(batch_size, sequence_length)` -- integers
- Output shape: `(batch_size, sequence_length, embedding_dim)` -- dense vectors

In [ ]:
# Demonstrate the Embedding layer
EMBEDDING_DIM = 128

embedding_layer = keras.layers.Embedding(
    input_dim=MAX_TOKENS,
    output_dim=EMBEDDING_DIM,
)

# Pass the encoded sample through the embedding
embedded = embedding_layer(encoded)

print(f"Input shape (encoded):   {encoded.shape}")
print(f"Output shape (embedded): {embedded.shape}")
print(f"\nEmbedding for token '{vocab[encoded[0][0]]}' (first 10 dims):")
print(embedded[0, 0, :10].detach().numpy())

## 5. Build the Sentiment Classification Model

Our model architecture:

```
TextVectorization -> Embedding(10000, 128) -> GlobalAveragePooling1D -> Dense(1, sigmoid)
```

**GlobalAveragePooling1D** averages the embedding vectors across the sequence dimension, producing a single 128-dimensional vector per review. This is a simple but effective approach for text classification.

In [ ]:
# Build the model
model = keras.Sequential([
    keras.layers.Input(shape=(1,), dtype="string"),
    text_vectorizer,
    keras.layers.Embedding(input_dim=MAX_TOKENS, output_dim=EMBEDDING_DIM),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# Train the model
history = model.fit(
    x_train_text,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="Train Loss")
axes[0].plot(history.history["val_loss"], label="Val Loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train Accuracy")
axes[1].plot(history.history["val_accuracy"], label="Val Accuracy")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test data
test_loss, test_acc = model.evaluate(x_test_text, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Test on custom examples
test_reviews = [
    "This movie was absolutely terrible. I hated every minute.",
    "What a beautiful and moving film. Truly a masterpiece!",
    "It was okay, nothing special but not bad either.",
]

predictions = model.predict(np.array(test_reviews), verbose=0)

for review, pred in zip(test_reviews, predictions):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    print(f"\n{sentiment} ({pred[0]:.4f}): {review}")